# Lesson 1 Datasets and How to Use Them

Hello and welcome to this brief introductory course into transformers. Transformers are an incredibly powerful machine learning tool with a wide variety of uses in every field. However, before our model can learn we need to build some foundational tools. First up is the backbone of every machine learning model, **DATA**.

### Why does everyone care so much about data?

As we will come to learn, all machine learning models boil down to a maze of simple nodes doing basic math. When they start all of these nodes are random. However, by passing data through them and applying techniques like _gradient descent_, _attention_, and _token embeddings_ we can mold a collection of basic nodes into a complex predictive algorithm.

### So... how do we manage data

There are a million different tools for creating datasets. Each of which have their own advantages and disadvantages. In our case we will be using the <a href=https://huggingface.co/docs/datasets/index>Huggingface Datasets Class</a>. This class allows easy access to the thousands of free datasets already collected on the Huggingface website. It also allows to easily pass our data to the <a href=https://huggingface.co/docs/transformers/index>Transformers</a> library which we will use later on.


Lets get started.

In [ ]:
#First off lets import the load_dataset function

from datasets import load_dataset


### Now we can import our dataset

While there are quite a few possible fields to pass load_dataset we only care about 2 right now
1. path: this is the Huggingface identifier for the dataset.
    * ex. 'ErikCikalleshi/new_york_times_news_1987_1995'
2. split: this is the subset of the larger dataset we want to use. There are usually two options for each dataset.
    * 'train' is the larger subset used to train models. It usually holds about 80% of the data
    * 'test' is the smaller subset used to evaluation your model after training. When first making a model or practicing it is better to use test as it will save you the wait of downloading 100,000+ lines of data.

The examples given above are from a dataset holding every New York Times article from 1987-1995. Try loading the 'test' subset of this dataset by filling in the fields below.

In [ ]:
#import the new york times dataset
nyt_dataset = load_dataset(path='fake-path', split='fake-split')
print(nyt_dataset)

In [6]:
nyt_dataset = load_dataset(path='ErikCikalleshi/new_york_times_news_1987_1995', split='test')
print(nyt_dataset)

Dataset({
    features: ['date', 'title', 'content'],
    num_rows: 64651
})


### Above you should be able to see the features (aka. columns) of the dataset as well as how many rows there are.

Now try accesing some of the data to see how the dataset is structured

In [ ]:
#accesing any column by name will by default print out the entire column structure. 
#this is helpful to make sure everything downloaded, but not the most useful
print(nyt_dataset['date'])
print(nyt_dataset['title'])
print(nyt_dataset['content'])


Column([1992, 1988, 1989, 1987, 1987, ...])
Column(['HOME IMPROVEMENT', "Review/Television; 'Quiet Victory,' on Beating the Odds", 'CZECH AIDE DENIES VALIDITY OF PACT ON SOVIET TROOPS', 'City Water Agency Sets 7.017% Top Bond Yield', 'AN UNFUNNY PREMIERE', ...])
Column(["EVEN if your house is well insulated, unsealed cracks and other air leaks may be robbing you of comfort and raising your heating and cooling bills. Weatherstripping around doors and windows can help, but for the maximum benefit, spend a day or two scouting out and plugging leaks in less obvious places.\nTo detect air leaks in a building, professional energy auditors use instruments ranging from infrared scanners to tubes that emit thin streams of smoke. (To find an energy auditor, check the listings in the Yellow Pages under Energy Conservation Services, or call your local utility company.) You can test for leaks by holding thread or a strip of tissue paper near spots where outside air may be slipping inside. If the th

In [31]:
'''
Note that you can access an entire row by index and get a dictionary object holding the individual columns. Or you can split the dataset even further 

A dataset basically boils down to a fancy collection of dictionaries with a bunch of helper functions that make them very useful for training models.
'''

#first lets shuffle our dataset to make sure we get a random selection of rows
nyt_dataset = nyt_dataset.shuffle()

#then lets cut our 64.7k rows down to a more manageable size using the shard function
#this splits our dataset into equal pieces and allows us to pick one to keep
data = nyt_dataset.shard(num_shards=100, index=0)

print(f"Now our massive dataset has just : {len(data)} rows.")


#now lets take a look at just one row on it's own to see the structure
data[0]


Now our massive dataset has just : 647 rows.


{'date': 1992,
 'title': 'Market Place; Small Investors Stepping Aside?',
 'content': 'SIGNS of a retreat by small investors led to much wringing of hands yesterday on Wall Street, where some brokerage stocks took a nasty hit. The selling was prompted by an announcement by Charles Schwab & Company, the discount broker, that said late Wednesday that business had fallen to an average of 18,000 trades a day recently, down from boom levels of 32,300 in January.\nSchwab shares were down as much as $4.25, to $18.25, in early trading on the New York Stock Exchange yesterday, though they rebounded to close down $2.625 at $19.875. The shares of Quick & Reilly Group Inc., another discount broker, were also down, falling 62.5 cents, to $19.375, on the Big Board.\nSIGNS of a retreat by small investors led to much wringing of hands yesterday on Wall Street, where some brokerage stocks took a nasty hit. The selling was prompted by an announcement by Charles Schwab & Company, the discount broker, tha

### Quickly summarize tokenizers here

We'll go into more depth on tokenizers when we build our first model

# Dataloaders

Datasets are great for holding massive amounts of data in an organized way. However, in order to train a model we need to pull out groups of rows in order to pass them to our model. These are called batches. To do this we use another specialized object called a **Dataloader**

In order to use a dataloader properly we will need to build a custom dataset class. I know this may seem redundant given the **Dataset** object we just loaded but a custom dataset class gives us much more control over the data we feed our model. It also allows the **Dataloader** to handle things like batching for us.

In [ ]:
#first impor the dataloader class. You'll need pytorch installed for this: 'pip install pytorch'
from torch.utils.data import Dataset, DataLoader


In [ ]:
class NYT_Dataset(Dataset):
    def __init__(self, nyt_dataset, num_examples=None, tokenizer=None):
        self.dataset = nyt_dataset
        self.num_examples = len(self.dataset)
        self.tokenizer = tokenizer

    def __len__(self):
        return self.num_examples
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        #show reformatting here
        return item


In [ ]:
#dataloader part goes here